In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact

from statsmodels.stats.multitest import multipletests

import re

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = ['C29', 'C37', 'C38', 'C39']
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
leiden_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', header=0, index_col=0)
leiden['donor_leiden'] = [f'{x}_{y}' for x,y in zip(leiden['line'], leiden['renamed_leiden'])]
leiden.shape

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
clusters['donor_leiden'] = leiden.reindex(clusters.index)['donor_leiden']
tmp_clusters = clusters[~clusters['donor_leiden'].isna()].copy()
tmp_clusters['donor_leiden'] = tmp_clusters['donor_leiden'].replace('C39_Inter_2', 'C39_Fibro_15')
tmp_clusters.shape

In [ ]:
s = 'Inter2'
# For each donor, for each mc cluster: 2x2 Fisher test (cluster vs not, in-cluster vs out-of-cluster)
results = []
for d in donors:
    donor_cells = tmp_clusters[tmp_clusters['donor'] == d]
    is_cluster = donor_cells['pred_coarse'] == s
    
    mask = ['Fibro' in x for x in donor_cells['donor_leiden']]
    donor_cells = donor_cells[mask].copy()
    for mc in sorted(donor_cells['donor_leiden'].unique()):
        in_cluster = donor_cells['donor_leiden'] == mc
        
        a = ( in_cluster &  is_cluster).sum()   # in cluster, cluster
        b = ( in_cluster & ~is_cluster).sum()   # in cluster, not cluster
        c = (~in_cluster &  is_cluster).sum()   # out of cluster, cluster
        e = (~in_cluster & ~is_cluster).sum()   # out of cluster, not cluster
        
        odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')
        results.append({
            'donor': d,
            'mc_cluster': mc,
            'in_cluster': a,
            'in_not_cluster': b,
            'out_cluster': c,
            'out_not_cluster': e,
            'odds_ratio': odds_ratio,
            'pval': pval,
        })

fisher_results = pd.DataFrame(results)
fisher_results['cluster'] = ['_'.join(x.split('_')[1:]) for x in fisher_results['mc_cluster']]
fisher_results.shape

In [ ]:
fisher_results.head()

In [ ]:
_, pval_adj, _, _ = multipletests(fisher_results['pval'], alpha=0.05, method='fdr_bh')
fisher_results['pval_adj'] = pval_adj
fisher_results['neglog10p'] = -1 * np.log10(fisher_results['pval_adj'])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i,ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = fisher_results[fisher_results['donor'] == d]
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', ax=ax)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
fisher_results.to_csv(f'{outdir}/snmct_mc_cluster_fisher.csv', sep='\t')

### what about fail1/fail2?

In [ ]:
def cluster_sort_key(name):
    # extract trailing number from names like "Early_10" -> 10
    match = re.search(r'(\d+)$', name)
    return int(match.group()) if match else 0

### add fold change

In [ ]:
s = 'Inter1'
PSEUDOCOUNT = 0.5  # avoids div-by-zero / log2(0) for rare/absent categories

results = []
for d in donors:
    donor_cells = tmp_clusters[tmp_clusters['donor'] == d]
    is_cluster = donor_cells['pred_coarse'] == s
    
    mask = ['Fibro' in x for x in donor_cells['donor_leiden']]
    donor_cells = donor_cells[mask].copy()
    for mc in sorted(donor_cells['donor_leiden'].unique()):
        in_cluster = donor_cells['donor_leiden'] == mc
        
        a = ( in_cluster &  is_cluster).sum()   # in cluster, cluster
        b = ( in_cluster & ~is_cluster).sum()   # in cluster, not cluster
        c = (~in_cluster &  is_cluster).sum()   # out of cluster, cluster
        e = (~in_cluster & ~is_cluster).sum()   # out of cluster, not cluster
        
        odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')

        # proportion of `s` inside vs outside the cluster, log2 fold change
        prop_in  = (a + PSEUDOCOUNT) / (a + b + 2 * PSEUDOCOUNT)
        prop_out = (c + PSEUDOCOUNT) / (c + e + 2 * PSEUDOCOUNT)
        log2fc = np.log2(prop_in / prop_out)

        results.append({
            'donor': d,
            'mc_cluster': mc,
            'in_cluster': a,
            'in_not_cluster': b,
            'out_cluster': c,
            'out_not_cluster': e,
            'odds_ratio': odds_ratio,
            'pval': pval,
            'prop_in': prop_in,
            'prop_out': prop_out,
            'log2fc': log2fc,
        })

inter1_results = pd.DataFrame(results)
inter1_results['cluster'] = ['_'.join(x.split('_')[1:]) for x in inter1_results['mc_cluster']]
_, pval_adj, _, _ = multipletests(inter1_results['pval'], alpha=0.05, method='fdr_bh')
inter1_results['pval_adj'] = pval_adj
inter1_results['neglog10p'] = -1 * np.log10(inter1_results['pval_adj'])
inter1_results['cluster'] = [x.replace('Fibro', 'Early') for x in inter1_results['cluster']]

# ---- significance plot (unchanged) ----
fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = inter1_results[inter1_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle(s, fontsize=20)

# ---- log2FC plot ----
fig2, axes2 = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes2.flatten()):
    d = donors[i]
    tmp_results = inter1_results[inter1_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', order=cluster_order, ax=ax)
    ax.axhline(y=0, color='black', linewidth=1)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig2.suptitle(f'{s} log2FC (in-cluster vs. out-of-cluster)', fontsize=16)
plt.show()

In [ ]:
s = 'Inter2'
PSEUDOCOUNT = 0.5  # avoids div-by-zero / log2(0) for rare/absent categories

results = []
for d in donors:
    donor_cells = tmp_clusters[tmp_clusters['donor'] == d]
    is_cluster = donor_cells['pred_coarse'] == s
    
    mask = ['Fibro' in x for x in donor_cells['donor_leiden']]
    donor_cells = donor_cells[mask].copy()
    for mc in sorted(donor_cells['donor_leiden'].unique()):
        in_cluster = donor_cells['donor_leiden'] == mc
        
        a = ( in_cluster &  is_cluster).sum()   # in cluster, cluster
        b = ( in_cluster & ~is_cluster).sum()   # in cluster, not cluster
        c = (~in_cluster &  is_cluster).sum()   # out of cluster, cluster
        e = (~in_cluster & ~is_cluster).sum()   # out of cluster, not cluster
        
        odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')

        # proportion of `s` inside vs outside the cluster, log2 fold change
        prop_in  = (a + PSEUDOCOUNT) / (a + b + 2 * PSEUDOCOUNT)
        prop_out = (c + PSEUDOCOUNT) / (c + e + 2 * PSEUDOCOUNT)
        log2fc = np.log2(prop_in / prop_out)

        results.append({
            'donor': d,
            'mc_cluster': mc,
            'in_cluster': a,
            'in_not_cluster': b,
            'out_cluster': c,
            'out_not_cluster': e,
            'odds_ratio': odds_ratio,
            'pval': pval,
            'prop_in': prop_in,
            'prop_out': prop_out,
            'log2fc': log2fc,
        })

inter2_results = pd.DataFrame(results)
inter2_results['cluster'] = ['_'.join(x.split('_')[1:]) for x in inter2_results['mc_cluster']]
_, pval_adj, _, _ = multipletests(inter2_results['pval'], alpha=0.05, method='fdr_bh')
inter2_results['pval_adj'] = pval_adj
inter2_results['neglog10p'] = -1 * np.log10(inter2_results['pval_adj'])
inter2_results['cluster'] = [x.replace('Fibro', 'Early') for x in inter2_results['cluster']]

# ---- significance plot (unchanged) ----
fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = inter2_results[inter2_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle(s, fontsize=20)

# ---- log2FC plot ----
fig2, axes2 = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes2.flatten()):
    d = donors[i]
    tmp_results = inter2_results[inter2_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', order=cluster_order, ax=ax)
    ax.axhline(y=0, color='black', linewidth=1)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig2.suptitle(f'{s} log2FC (in-cluster vs. out-of-cluster)', fontsize=16)
plt.show()

In [ ]:
s = 'Fail1'
PSEUDOCOUNT = 0.5  # avoids div-by-zero / log2(0) for rare/absent categories

results = []
for d in donors:
    donor_cells = tmp_clusters[tmp_clusters['donor'] == d]
    is_cluster = donor_cells['pred_coarse'] == s
    
    mask = ['Fibro' in x for x in donor_cells['donor_leiden']]
    donor_cells = donor_cells[mask].copy()
    for mc in sorted(donor_cells['donor_leiden'].unique()):
        in_cluster = donor_cells['donor_leiden'] == mc
        
        a = ( in_cluster &  is_cluster).sum()   # in cluster, cluster
        b = ( in_cluster & ~is_cluster).sum()   # in cluster, not cluster
        c = (~in_cluster &  is_cluster).sum()   # out of cluster, cluster
        e = (~in_cluster & ~is_cluster).sum()   # out of cluster, not cluster
        
        odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')

        # proportion of `s` inside vs outside the cluster, log2 fold change
        prop_in  = (a + PSEUDOCOUNT) / (a + b + 2 * PSEUDOCOUNT)
        prop_out = (c + PSEUDOCOUNT) / (c + e + 2 * PSEUDOCOUNT)
        log2fc = np.log2(prop_in / prop_out)

        results.append({
            'donor': d,
            'mc_cluster': mc,
            'in_cluster': a,
            'in_not_cluster': b,
            'out_cluster': c,
            'out_not_cluster': e,
            'odds_ratio': odds_ratio,
            'pval': pval,
            'prop_in': prop_in,
            'prop_out': prop_out,
            'log2fc': log2fc,
        })

fail1_results = pd.DataFrame(results)
fail1_results['cluster'] = ['_'.join(x.split('_')[1:]) for x in fail1_results['mc_cluster']]
_, pval_adj, _, _ = multipletests(fail1_results['pval'], alpha=0.05, method='fdr_bh')
fail1_results['pval_adj'] = pval_adj
fail1_results['neglog10p'] = -1 * np.log10(fail1_results['pval_adj'])
fail1_results['cluster'] = [x.replace('Fibro', 'Early') for x in fail1_results['cluster']]

# ---- significance plot (unchanged) ----
fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = fail1_results[fail1_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle(s, fontsize=20)

# ---- log2FC plot ----
fig2, axes2 = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes2.flatten()):
    d = donors[i]
    tmp_results = fail1_results[fail1_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', order=cluster_order, ax=ax)
    ax.axhline(y=0, color='black', linewidth=1)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig2.suptitle(f'{s} log2FC (in-cluster vs. out-of-cluster)', fontsize=16)
plt.show()

In [ ]:
s = 'Fail2'
PSEUDOCOUNT = 0.5  # avoids div-by-zero / log2(0) for rare/absent categories

results = []
for d in donors:
    donor_cells = tmp_clusters[tmp_clusters['donor'] == d]
    is_cluster = donor_cells['pred_coarse'] == s
    
    mask = ['Fibro' in x for x in donor_cells['donor_leiden']]
    donor_cells = donor_cells[mask].copy()
    for mc in sorted(donor_cells['donor_leiden'].unique()):
        in_cluster = donor_cells['donor_leiden'] == mc

        a = ( in_cluster &  is_cluster).sum()   # in cluster, cluster
        b = ( in_cluster & ~is_cluster).sum()   # in cluster, not cluster
        c = (~in_cluster &  is_cluster).sum()   # out of cluster, cluster
        e = (~in_cluster & ~is_cluster).sum()   # out of cluster, not cluster
        
        odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')

        # proportion of `s` inside vs outside the cluster, log2 fold change
        prop_in  = (a + PSEUDOCOUNT) / (a + b + 2 * PSEUDOCOUNT)
        prop_out = (c + PSEUDOCOUNT) / (c + e + 2 * PSEUDOCOUNT)
        log2fc = np.log2(prop_in / prop_out)

        results.append({
            'donor': d,
            'mc_cluster': mc,
            'in_cluster': a,
            'in_not_cluster': b,
            'out_cluster': c,
            'out_not_cluster': e,
            'odds_ratio': odds_ratio,
            'pval': pval,
            'prop_in': prop_in,
            'prop_out': prop_out,
            'log2fc': log2fc,
        })

fail2_results = pd.DataFrame(results)
fail2_results['cluster'] = ['_'.join(x.split('_')[1:]) for x in fail2_results['mc_cluster']]
_, pval_adj, _, _ = multipletests(fail2_results['pval'], alpha=0.05, method='fdr_bh')
fail2_results['pval_adj'] = pval_adj
fail2_results['neglog10p'] = -1 * np.log10(fail2_results['pval_adj'])
fail2_results['cluster'] = [x.replace('Fibro', 'Early') for x in fail2_results['cluster']]

# ---- significance plot (unchanged) ----
fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = fail2_results[fail2_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle(s, fontsize=20)

# ---- log2FC plot ----
fig2, axes2 = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes2.flatten()):
    d = donors[i]
    tmp_results = fail2_results[fail2_results['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', order=cluster_order, ax=ax)
    ax.axhline(y=0, color='black', linewidth=1)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig2.suptitle(f'{s} log2FC (in-cluster vs. out-of-cluster)', fontsize=16)
plt.show()

In [ ]:
fail1_results['rna_cluster'] = 'Fail1'
fail2_results['rna_cluster'] = 'Fail2'
inter2_results['rna_cluster'] = 'Inter2'

merged = pd.concat([fail1_results, fail2_results, inter2_results])
merged.shape

In [ ]:
merged.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = merged[merged['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', hue='rna_cluster', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
    
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle('FDR-corrected -log10 p-values from Fisher test', fontsize=16)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = merged[merged['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', hue='rna_cluster', order=cluster_order, ax=ax)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
    
    
plt.subplots_adjust(wspace=0.4, hspace=0.4)
fig.suptitle('FDR-corrected -log10 p-values from Fisher test', fontsize=16)

### include inter1

In [ ]:
fail1_results['rna_cluster'] = 'Fail1'
fail2_results['rna_cluster'] = 'Fail2'
inter1_results['rna_cluster'] = 'Inter1'
inter2_results['rna_cluster'] = 'Inter2'

merged = pd.concat([fail1_results, fail2_results, inter1_results, inter2_results])
merged.shape

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = merged[merged['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='neglog10p', hue='rna_cluster', order=cluster_order, ax=ax)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1)
    ax.set_ylim(0, 15)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
    
plt.subplots_adjust(wspace=0.2, hspace=0.4)
fig.suptitle('FDR-corrected -log10 p-values from Fisher test', fontsize=16)

fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_results = merged[merged['donor'] == d]
    cluster_order = sorted(tmp_results['cluster'].unique(), key=cluster_sort_key)
    sns.barplot(data=tmp_results, x='cluster', y='log2fc', hue='rna_cluster', order=cluster_order, ax=ax)
    ax.set_title(d)
    ax.tick_params(axis='x', rotation=45)
    
    
plt.subplots_adjust(wspace=0.4, hspace=0.4)
fig.suptitle('FDR-corrected -log10 p-values from Fisher test', fontsize=16)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged.to_csv(f'{outdir}/snmct_donor_fibro_fisher_fail_inter.csv', sep='\t')

### now I want a fisher by donor

In [ ]:
mask = [x.split('_')[1] == 'Fibro' for x in tmp_clusters['donor_leiden']]
fibro_clusters = tmp_clusters[mask].copy()
fibro_clusters['cluster'] = [x.split('_')[1] for x in fibro_clusters['donor_leiden']]
fibro_clusters.head()

In [ ]:
results = []
for d in donors:
    in_donor = fibro_clusters['donor'] == d
    is_inter2 = fibro_clusters['pred_coarse'] == 'Inter2'

    a = ( in_donor &  is_inter2).sum()
    b = ( in_donor & ~is_inter2).sum()
    c = (~in_donor &  is_inter2).sum()
    e = (~in_donor & ~is_inter2).sum()

    odds_ratio, pval = fisher_exact([[a, b], [c, e]], alternative='greater')
    results.append({
        'donor': d,
        'in_inter2': a,
        'in_not_inter2': b,
        'out_inter2': c,
        'out_not_inter2': e,
        'odds_ratio': odds_ratio,
        'pval': pval,
    })

fisher_results = pd.DataFrame(results)
fisher_results.shape

In [ ]:
_, pval_adj, _, _ = multipletests(fisher_results['pval'], alpha=0.05, method='fdr_bh')
fisher_results['pval_adj'] = pval_adj
fisher_results['neglog10p'] = -1 * np.log10(fisher_results['pval_adj'])

In [ ]:
fisher_results

In [ ]:
fig, ax = plt.subplots(1, figsize=(4,4))

sns.barplot(data=fisher_results, x='donor', y='neglog10p', fill='donor', ax=ax)

### literally just the ratio

In [ ]:
mask = [x.split('_')[1] == 'Fibro' for x in tmp_clusters['donor_leiden']]
fibro_clusters = tmp_clusters[mask].copy()
fibro_clusters['cluster'] = [x.split('_')[1] for x in fibro_clusters['donor_leiden']]
fibro_clusters.shape

In [ ]:
fibro_clusters.head()

In [ ]:
clust_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
fibro_counts = fibro_clusters.groupby(['donor', 'pred_coarse']).size().unstack(fill_value=0)
fibro_dist = fibro_counts.div(fibro_counts.sum(axis=1), axis=0).T.loc[clust_order]
fibro_dist.sum(axis=0)

fibro_dist['cluster'] = fibro_dist.index
to_plot = fibro_dist.melt(id_vars='cluster', value_vars=donors)
mask = to_plot['cluster'] == 'Inter2'
to_plot = to_plot[mask]
sns.barplot(data=to_plot, x='donor', y='value', fill='donor')

In [ ]:
fibro_dist.head()

In [ ]:
fibro_dist['cluster'] = fibro_dist.index
to_plot = fibro_dist.melt(id_vars='cluster', value_vars=donors)
mask = to_plot['cluster'] == 'Inter2'
to_plot = to_plot[mask]
sns.barplot(data=to_plot, x='donor', y='value', fill='donor')

In [ ]:
s = 'Fail1'
fibro_dist['cluster'] = fibro_dist.index
to_plot = fibro_dist.melt(id_vars='cluster', value_vars=donors)
mask = to_plot['cluster'] == s
to_plot = to_plot[mask]
ax = sns.barplot(data=to_plot, x='donor', y='value', fill='donor')
ax.set_title(s)
ax.set_ylabel('Nuclei (%)')

In [ ]:
s = 'Fail2'
fibro_dist['cluster'] = fibro_dist.index
to_plot = fibro_dist.melt(id_vars='cluster', value_vars=donors)
mask = to_plot['cluster'] == s
to_plot = to_plot[mask]
ax = sns.barplot(data=to_plot, x='donor', y='value', fill='donor')
ax.set_title(s)
ax.set_ylabel('Nuclei (%)')

In [ ]:
fibro_dist['cluster'] = fibro_dist.index
merged = fibro_dist.melt(id_vars='cluster', value_vars=donors)
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged.to_csv(f'{outdir}/snmct_donor_fibro_fisher_all_clusters.csv', sep='\t')

In [ ]:
!date